# Part 1: Best-of-N with LLaMA-3.1 (naive reward)
Naive reward only counts number of citations, which is anything inside square brackets [].

1. Config

In [ ]:
import json
import pandas as pd
from vllm import LLM, SamplingParams
import re
from pathlib import Path
from transformers import AutoTokenizer

# Config 
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct" 
DATA_PATH = "data/train.json" 
OUTPUT_DIR = Path("results/ablations")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

2. Reward function

In [ ]:
def base_reward(text: str, question: str, reference: str) -> float:
    # Matches [http://...] or [https://...] or [www...]
    link_pattern = r'\[(?:https?://|www\.)[^\s\]]+\]'
    links = re.findall(link_pattern, text)
    
    # Reward based on link count (with a cap to prevent infinite looping/hacking)
    return float(min(len(links), 5))

3. Initialization

In [ ]:
# Initialization
print("Loading model and tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm = LLM(model=MODEL_NAME, dtype="bfloat16", device="cuda")

4. Main pipeline

In [ ]:
def run_ablation_batch(temp: float, top_p: float, n_samples: int = 16, max_prompts: int = None):
    """
    Runs the full dataset through the Best of N pipeline with specific 
    decoding parameters to observe impact on reward hacking.
    """

    # 0. Define sampling parameters for this specific run
    sampling_params = SamplingParams(
        n=n_samples,
        temperature=temp,
        top_p=top_p,
        max_tokens=300,
        repetition_penalty=1.1, # Penalize looping
        stop_token_ids=[
            tokenizer.eos_token_id, 
            tokenizer.convert_tokens_to_ids("<|eot_id|>") # LLaMA 3 specific turn-end token
        ]
    )

    # 1. Load Dataset and Format Prompts, Slice
    with open(DATA_PATH, 'r') as f:
        raw_data = json.load(f) # Assumes list of dicts with "instruction" and "best_answer"

    # Slice
    if max_prompts:
        raw_data = raw_data[:max_prompts]
        print(f"Limiting execution to the first {max_prompts} prompts.")

    formatted_prompts = []
    for item in raw_data:
        messages = [
            {"role": "system", "content": "You are a helpful research assistant. You MUST include inline citations in your response using the exact format: [http://example.com]. Ensure your claims are well-supported."},
            
            # --- Few-Shot Example 1 ---
            {"role": "user", "content": "What are the primary colors of light?"},
            {"role": "assistant", "content": "The primary colors of light are red, green, and blue [http://en.wikipedia.org/wiki/Primary_color]. When these three colors are combined in various ways, they can produce a wide spectrum of other colors."},
            
            # --- Few-Shot Example 2 ---
            {"role": "user", "content": "Who wrote the novel '1984'?"},
            {"role": "assistant", "content": "The dystopian novel '1984' was written by the British author George Orwell [http://en.wikipedia.org/wiki/Nineteen_Eighty-Four]. It was published in 1949 and focuses on themes of totalitarianism and surveillance [http://www.sparknotes.com/lit/1984/themes/]."},
            
            # --- The Actual Target Prompt ---
            {"role": "user", "content": item['instruction']}
        ]
        
        # Apply the template
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        formatted_prompts.append(prompt_text)

    # 2. Batch Inference
    print(f"Running Ablation: Temp={temp}, Top-P={top_p}...")
    print(f"Generating {len(formatted_prompts) * n_samples} total responses...")
    outputs = llm.generate(formatted_prompts, sampling_params)

    # 3. Best-of-N Selection 
    final_results = []
    
    for i, prompt_output in enumerate(outputs):
        best_response = None
        max_reward = -1.0

        question = raw_data[i]['instruction']
        reference = raw_data[i].get('best_answer', '')
        
        # Evaluate all N samples for this specific prompt
        for candidate in prompt_output.outputs:
            
            score = reward_fn(candidate.text, question, reference)
            
            if score > max_reward:
                max_reward = score
                best_response = candidate.text
        
        print(f"Highest Reward Achieved: {max_reward}")
        print(f"Winning Response Snippet: {best_response[:100]}...\n")
        print("-" * 40)
                
        # Save the "Winner" along with metadata for Student B
        final_results.append({
            "question": question,
            "reference": reference, 
            "model_output": best_response,
            "reward_score": max_reward,
            "params": {"temp": temp, "top_p": top_p, "n_val": n_samples}
        })


    # 4. Save to JSON for Evaluation Script
    filename = f"gen_t{temp}_p{top_p}_n{n_samples}.json"
    with open(OUTPUT_DIR / filename, 'w') as f:
        json.dump(final_results, f, indent=4)
    
    print(f"Saved results to {OUTPUT_DIR / filename}")

5. Execution

In [ ]:
configs = [
    # {"temp": 0.2, "p": 0.9},
    {"temp": 0.7, "p": 0.9},  # The baseline
    # {"temp": 0.8, "p": 0.95},
    # {"temp": 1.2, "p": 1.0}
]

for config in configs:
    run_ablation_batch(temp=config["temp"], top_p=config["p"], reward_fn=base_reward, max_prompts=200)

Evaluate the results


In [ ]:
with open("gen_t0.7_p0.9_n16.json", "r") as f:
    results = json.load(f)

# Run evaluation script on "results" to compute final metrics and analyze reward hacking patterns.

# Part 2: Best-of-N with Llama-3.1 (shaped reward)

In [ ]:
import extract_urls
import verify_source_helper as vsh

Shaped reward function: [insert description]

In [ ]:
def shaped_reward(generated_text, question, reference_answer) -> tuple[float, list[dict]]:
    """
    Formula:
        quantity_bonus = n_cited / MAX_CITED_SOURCES   # in [0, 1]
        mean_quality   = average verify_source score   # in [0, 1]
        reward         = mean_quality x quantity_bonus

    Returns: 
        a tuple containing:
        1. the reward score 
        2. a list of URLs with their individual quality scores
    """
    urls = extract_urls.extract_and_normalise_urls(generated_text)

    if not urls:
        return 0.0, []

    per_url: list[dict] = []
    quality_scores: list[float] = []

    for url in urls:
        try:
            raw   = vsh.verify_source(url, question, reference_answer)
            score = float(raw) if isinstance(raw, (int, float)) else 0.0
        except Exception:
            score = 0.0

        quality_scores.append(score)
        per_url.append({"url": url, "score": score})

    mean_quality   = sum(quality_scores) / len(quality_scores)
    quantity_bonus = len(urls) / MAX_CITED_SOURCES
    reward         = mean_quality * quantity_bonus

    return round(reward, 4), per_url